# Custom Functions: Development to Production

This tutorial demonstrates the complete workflow from prototyping custom functions in a notebook to deploying them in production via the CLI.

## What You'll Learn

1. **Prototyping Phase**: Define config classes and use `NatFunction(config=..., registered_function=...)` for rapid iteration
2. **Production Phase**: Export functions with `%%writefile` to a proper Python module
3. **Deployment**: Install the package and run via CLI

## The Factory Pattern

Instead of creating custom subclasses, we use the **factory pattern**:

```python
# Old way (creating subclasses):
class Power(PowerConfig, NatFunction):
    pass
power_tool = Power(name="power", registered_function=build_fn)

# New way (factory pattern - no subclass needed):
power_tool = NatFunction(config=PowerConfig(), name="power", registered_function=build_fn)
```

## The Math Agent

We'll build a comprehensive math agent with:
- **Power**: Exponentiation (base^exponent)
- **SquareRoot**: Calculate square roots
- **AdvancedMathGroup**: Function group with factorial, logarithm, trigonometry, and statistics


In [1]:
import sys
from pathlib import Path

# Setup paths for development
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


✅ Environment configured


---

# Part 1: Prototyping with the Factory Pattern

In this section, we define config classes and use `NatFunction(config=..., registered_function=...)` to register them dynamically. This allows rapid prototyping without creating separate files or subclasses.

## Step 1.1: Define the Power Function


In [2]:
import math
import warnings

from nat.builder.builder import Builder
from nat.builder.function_info import FunctionInfo
from nat.data_models.function import FunctionBaseConfig
from nat.utils.sdk.nat_function import NatFunction


# Define the Power function configuration (no subclass needed!)
class PowerConfig(FunctionBaseConfig, name="power"):
    """Raise a base number to an exponent power."""


# Define the build function
async def power_build_fn(config: PowerConfig, builder: Builder):
    """Build function for the power operation."""

    async def _power(base: float, exponent: float) -> float:
        """Raise base to the power of exponent. Returns base^exponent."""
        return math.pow(base, exponent)

    yield FunctionInfo.from_fn(
        _power,
        description="Raise a base number to an exponent power (e.g., 2^3 = 8)."
    )


# Create the tool using the factory pattern - no subclass needed!
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    power_tool = NatFunction(
        config=PowerConfig(),
        name="power",
        registered_function=power_build_fn
    )

print(f"✅ Created: {power_tool.computed_name}")


✅ Created: power


## Step 1.2: Define the Square Root Function


In [3]:
# Define the SquareRoot function configuration (no subclass needed!)
class SquareRootConfig(FunctionBaseConfig, name="square_root"):
    """Calculate the square root of a number."""


async def sqrt_build_fn(config: SquareRootConfig, builder: Builder):
    """Build function for the square root operation."""

    async def _sqrt(number: float) -> float:
        """Calculate the square root of a number. Returns √number."""
        if number < 0:
            raise ValueError("Cannot calculate square root of negative number")
        return math.sqrt(number)

    yield FunctionInfo.from_fn(
        _sqrt,
        description="Calculate the square root of a number (e.g., √16 = 4)."
    )


# Create the tool using the factory pattern
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sqrt_tool = NatFunction(
        config=SquareRootConfig(),
        name="square_root",
        registered_function=sqrt_build_fn
    )

print(f"✅ Created: {sqrt_tool.computed_name}")


✅ Created: square_root


## Step 1.3: Define the Advanced Math Function Group

Function groups package multiple related functions together, sharing configuration and resources.


In [4]:
from typing import Literal

import numpy as np
from pydantic import Field

from nat.builder.function import FunctionGroup
from nat.data_models.function import FunctionGroupBaseConfig
from nat.utils.sdk.nat_function_group import NatFunctionGroup


# Define the function group configuration (no subclass needed!)
class AdvancedMathGroupConfig(FunctionGroupBaseConfig, name="advanced_math"):
    """
    A group of advanced mathematical functions using NumPy.

    Provides: factorial, logarithm, trigonometry, and statistics.
    """
    precision: int = Field(
        default=6, description="Number of decimal places for results"
    )


async def advanced_math_build_fn(config: AdvancedMathGroupConfig, builder: Builder):
    """Build the advanced math function group."""

    precision = config.precision
    group = FunctionGroup(config=config)

    async def _factorial(n: int) -> int:
        """Calculate the factorial of n (n!). Only works for non-negative integers."""
        if n < 0:
            raise ValueError("Factorial is not defined for negative numbers")
        return int(math.factorial(n))

    async def _logarithm(number: float, base: float = 10.0) -> float:
        """Calculate the logarithm of a number. Default is base 10 (common log)."""
        if number <= 0:
            raise ValueError("Logarithm is not defined for non-positive numbers")
        result = math.log(number) / math.log(base)
        return round(result, precision)

    async def _trigonometry(
        angle_degrees: float,
        function: Literal["sin", "cos", "tan"] = "sin",
    ) -> float:
        """Calculate sin, cos, or tan of an angle in degrees."""
        angle_radians = np.radians(angle_degrees)
        if function == "sin":
            result = float(np.sin(angle_radians))
        elif function == "cos":
            result = float(np.cos(angle_radians))
        elif function == "tan":
            result = float(np.tan(angle_radians))
        else:
            raise ValueError(f"Unknown function: {function}")
        return round(result, precision)

    async def _statistics(
        numbers: list[float],
        operation: Literal["mean", "median", "std"] = "mean",
    ) -> float:
        """Calculate mean, median, or standard deviation of a list of numbers."""
        arr = np.array(numbers)
        if operation == "mean":
            result = float(np.mean(arr))
        elif operation == "median":
            result = float(np.median(arr))
        elif operation == "std":
            result = float(np.std(arr))
        else:
            raise ValueError(f"Unknown operation: {operation}")
        return round(result, precision)

    # Add functions to the group
    group.add_function("factorial", _factorial, description="Calculate the factorial of n (n!)")
    group.add_function("logarithm", _logarithm, description="Calculate logarithm (default base 10)")
    group.add_function("trigonometry", _trigonometry, description="Calculate sin, cos, or tan")
    group.add_function("statistics", _statistics, description="Calculate mean, median, or std")

    yield group


# Create the function group using the factory pattern
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    advanced_math = NatFunctionGroup(
        config=AdvancedMathGroupConfig(precision=4),
        name="advanced_math",
        registered_function=advanced_math_build_fn,
    )

print(f"✅ Created function group: {advanced_math.computed_name}")


✅ Created function group: advanced_math


## Step 1.4: Create the Math Agent


In [5]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Combine all custom tools
tools = [power_tool, sqrt_tool, advanced_math]

# Create the agent
math_agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    additional_instructions=(
        "You are a math expert. Use the available tools to solve mathematical problems. "
        "Show your work step by step."
    ),
)

# Create the workflow
workflow = NatWorkflow(entrypoint=math_agent)

print(f"✅ Math Agent created with {len(tools)} tool(s):"
      f"\n   - {power_tool.computed_name}"
      f"\n   - {sqrt_tool.computed_name}"
      f"\n   - {advanced_math.computed_name} (function group)")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Math Agent created with 3 tool(s):
   - power
   - square_root
   - advanced_math (function group)


## Step 1.5: Test the Agent (Prototyping Phase)

Now let's test our inline functions to make sure they work before exporting to production:


In [6]:
# Test: Power function
result = await workflow.prompt("What is 2 raised to the power of 10?")
print(result)


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


1024.0


In [7]:
# Test: Square root function
result = await workflow.prompt("What is the square root of 144?")
print(result)


The square root of 144 is 12.0


In [8]:
# Test: Advanced math - factorial
result = await workflow.prompt("What is 5 factorial?")
print(result)


5 factorial is 120, which is often denoted as 5! = 120.


In [9]:
# Test: Advanced math - trigonometry
result = await workflow.prompt("What is the sine of 30 degrees?")
print(result)


0.5


In [10]:
# Test: Advanced math - statistics
result = await workflow.prompt("What is the mean of [10, 20, 30, 40, 50]?")
print(result)


The mean of the given list [10, 20, 30, 40, 50] is 30.0.


---

# Part 2: Export to Production

Now that prototyping is complete, we export our functions to a proper Python module using `%%writefile`.

**Key differences from prototyping:**
1. We add `@register_function` and `@register_function_group` decorators for automatic registration
2. We create subclasses (e.g., `class Power(PowerConfig, NatFunction)`) so users can import them
3. No `registered_function` parameter needed - decorators handle registration

> 💡 **Note**: In production, we use subclasses so users can `from math_tools import Power` and instantiate `Power()` directly. The factory pattern (`NatFunction(config=...)`) is best for rapid prototyping in notebooks.

## Step 2.1: Create the Package Directory


In [11]:
# Create the package directory
package_dir = Path("./math_tools/math_tools")
package_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ Created package directory: {package_dir}")


✅ Created package directory: math_tools/math_tools


## Step 2.2: Write the `__init__.py`


In [12]:
%%writefile math_tools/math_tools/__init__.py
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

"""Math tools package for the SDK tutorial."""

from .register import AdvancedMathGroup
from .register import AdvancedMathGroupConfig
from .register import Power
from .register import PowerConfig
from .register import SquareRoot
from .register import SquareRootConfig

__all__ = [
    "AdvancedMathGroup",
    "AdvancedMathGroupConfig",
    "Power",
    "PowerConfig",
    "SquareRoot",
    "SquareRootConfig",
]


Overwriting math_tools/math_tools/__init__.py


## Step 2.3: Write the `register.py` (Production Code)

This is the same code as above, but with:
1. Proper imports at the top
2. `@register_function` and `@register_function_group` decorators
3. No `registered_function` parameter needed


In [13]:
%%writefile math_tools/math_tools/register.py
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
"""
Math tools for the SDK tutorial - Production Version.

This module is auto-generated from 14_custom_functions_inline.ipynb.
The @register_function and @register_function_group decorators automatically
register these functions when the module is imported as a NAT plugin.
"""

import math
from collections.abc import AsyncGenerator
from typing import Literal

import numpy as np
from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.function import FunctionGroup
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.cli.register_workflow import register_function_group
from nat.data_models.function import FunctionBaseConfig
from nat.data_models.function import FunctionGroupBaseConfig
from nat.utils.sdk.nat_function import NatFunction
from nat.utils.sdk.nat_function_group import NatFunctionGroup


# =============================================================================
# Power Function
# =============================================================================

class PowerConfig(FunctionBaseConfig, name="power"):
    """Raise a base number to an exponent power."""


class Power(PowerConfig, NatFunction):
    """Power function for SDK use."""


@register_function(config_type=PowerConfig)
async def power_fn(
    _config: PowerConfig, _builder: Builder
) -> AsyncGenerator[FunctionInfo, None]:
    """Build the power function."""

    async def _power(base: float, exponent: float) -> float:
        """Raise base to the power of exponent. Returns base^exponent."""
        return math.pow(base, exponent)

    yield FunctionInfo.from_fn(
        _power, description="Raise a base number to an exponent power (e.g., 2^3 = 8)."
    )


# =============================================================================
# Square Root Function
# =============================================================================

class SquareRootConfig(FunctionBaseConfig, name="square_root"):
    """Calculate the square root of a number."""


class SquareRoot(SquareRootConfig, NatFunction):
    """Square root function for SDK use."""


@register_function(config_type=SquareRootConfig)
async def square_root_fn(
    _config: SquareRootConfig, _builder: Builder
) -> AsyncGenerator[FunctionInfo, None]:
    """Build the square root function."""

    async def _sqrt(number: float) -> float:
        """Calculate the square root of a number. Returns √number."""
        if number < 0:
            raise ValueError("Cannot calculate square root of negative number")
        return math.sqrt(number)

    yield FunctionInfo.from_fn(
        _sqrt, description="Calculate the square root of a number (e.g., √16 = 4)."
    )


# =============================================================================
# Advanced Math Function Group
# =============================================================================

class AdvancedMathGroupConfig(FunctionGroupBaseConfig, name="advanced_math"):
    """
    A group of advanced mathematical functions using NumPy.

    Provides: factorial, logarithm, trigonometry, and statistics.
    """
    precision: int = Field(
        default=6, description="Number of decimal places for results"
    )


class AdvancedMathGroup(AdvancedMathGroupConfig, NatFunctionGroup):
    """Advanced Math function group for SDK use."""


@register_function_group(config_type=AdvancedMathGroupConfig)
async def advanced_math_group(
    config: AdvancedMathGroupConfig, _builder: Builder
) -> AsyncGenerator[FunctionGroup, None]:
    """Build the advanced math function group."""

    precision = config.precision
    group = FunctionGroup(config=config)

    async def _factorial(n: int) -> int:
        """Calculate the factorial of n (n!). Only works for non-negative integers."""
        if n < 0:
            raise ValueError("Factorial is not defined for negative numbers")
        return int(math.factorial(n))

    async def _logarithm(number: float, base: float = 10.0) -> float:
        """Calculate the logarithm of a number. Default is base 10 (common log)."""
        if number <= 0:
            raise ValueError("Logarithm is not defined for non-positive numbers")
        if base <= 0 or base == 1:
            raise ValueError("Base must be positive and not equal to 1")
        result = math.log(number) / math.log(base)
        return round(result, precision)

    async def _trigonometry(
        angle_degrees: float,
        function: Literal["sin", "cos", "tan"] = "sin",
    ) -> float:
        """Calculate sin, cos, or tan of an angle in degrees."""
        angle_radians = np.radians(angle_degrees)
        if function == "sin":
            result = float(np.sin(angle_radians))
        elif function == "cos":
            result = float(np.cos(angle_radians))
        elif function == "tan":
            result = float(np.tan(angle_radians))
        else:
            raise ValueError(f"Unknown function: {function}")
        return round(result, precision)

    async def _statistics(
        numbers: list[float],
        operation: Literal["mean", "median", "std"] = "mean",
    ) -> float:
        """Calculate mean, median, or standard deviation of a list of numbers."""
        arr = np.array(numbers)
        if operation == "mean":
            result = float(np.mean(arr))
        elif operation == "median":
            result = float(np.median(arr))
        elif operation == "std":
            result = float(np.std(arr))
        else:
            raise ValueError(f"Unknown operation: {operation}")
        return round(result, precision)

    # Add functions to the group
    group.add_function("factorial", _factorial, description="Calculate the factorial of n (n!)")
    group.add_function("logarithm", _logarithm, description="Calculate logarithm (default base 10)")
    group.add_function("trigonometry", _trigonometry, description="Calculate sin, cos, or tan")
    group.add_function("statistics", _statistics, description="Calculate mean, median, or std")

    yield group


Overwriting math_tools/math_tools/register.py


---

# Part 3: Install and Run via CLI

> ⚠️ **Important**: Before running Part 3, **restart your kernel** to clear the inline function registrations from Part 1. This prevents naming conflicts between the prototype functions (`__main__/power`) and the production module functions (`math_tools/power`).
>
> In a real workflow, you would:
> 1. Prototype in the notebook (Part 1)
> 2. Export to a module (Part 2)
> 3. **Restart kernel**, install the package, and use the production classes (Part 3)

## Step 3.1: Install the Package


In [14]:
# Install the package (uncomment to run)
!cd math_tools && uv pip install -e .

print("To install the math tools package, run:")
print("  cd examples/notebooks/sdk/math_tools")
print("  uv pip install -e .")


Using Python 3.13.5 environment at: /Users/spastoriza/Documents/Programming/public/nat-fork/.venv
Resolved 135 packages in 5.25s                                       ⠋ Resolving dependencies...                                                     
   Building nat-math-tools @ file:///Users/spastoriza/Documents/Programming/publ
   Building nat-math-tools @ file:///Users/spastoriza/Documents/Programming/publ
   Building nvidia-nat @ file:///Users/spastoriza/Documents/Programming/public/n
   Building nat-math-tools @ file:///Users/spastoriza/Documents/Programming/publ
   Building nvidia-nat @ file:///Users/spastoriza/Documents/Programming/public/n
   Building nat-math-tools @ file:///Users/spastoriza/Documents/Programming/publ
   Building nvidia-nat @ file:///Users/spastoriza/Documents/Programming/public/n
   Building nat-math-tools @ file:///Users/spastoriza/Documents/Programming/publ
   Building nvidia-nat @ file:///Users/spastoriza/Documents/Programming/public/n
      Built nat-math-to

## Step 3.2: Create Production Workflow

After installing, we can import the production classes and create a workflow:


In [15]:
import sys
from pathlib import Path

# Setup paths for development (after kernel restart)
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Add the math_tools package directory to path (not the project root)
# This ensures we import from math_tools/math_tools/ not math_tools/
math_tools_pkg_path = Path("./math_tools").resolve()
if str(math_tools_pkg_path) not in sys.path:
    sys.path.insert(0, str(math_tools_pkg_path))

# Import NAT components
# Import production classes from math_tools package
from math_tools import AdvancedMathGroup  # pyright: ignore[reportAttributeAccessIssue]  # noqa: E402
from math_tools import Power  # pyright: ignore[reportAttributeAccessIssue]  # noqa: E402
from math_tools import SquareRoot  # pyright: ignore[reportAttributeAccessIssue]  # noqa: E402

from nat.agent.sdk import NatReActAgent  # noqa: E402
from nat.llm.sdk import NimLLM  # noqa: E402
from nat.utils.sdk.nat_workflow import NatWorkflow  # noqa: E402

# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Create production tools (no registered_function needed - decorators handle it)
prod_power = Power(name="power")
prod_sqrt = SquareRoot(name="square_root")
prod_advanced_math = AdvancedMathGroup(name="advanced_math", precision=4)

# Create production workflow
prod_agent = NatReActAgent(
    tools=[prod_power, prod_sqrt, prod_advanced_math],
    llm=llm,
    verbose=True,
    additional_instructions=(
        "You are a math expert. Use the available tools to solve mathematical problems."
    ),
)

prod_workflow = NatWorkflow(entrypoint=prod_agent)

print("✅ Production workflow created")


✅ Production workflow created


## Step 3.3: Save Configuration


In [16]:
# Create configs directory and save
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "math_agent.yaml"
prod_workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


Requested functions type `power` is ambiguous. Matched multiple functions by their local name: ['__main__/power', 'math_tools/power']. Please use the fully qualified functions name.
Available functions names:
 - __main__/power
 - __main__/square_root
 - nat.agent.react_agent/react_agent
 - nat_profiler_agent.tool/flow_chart
 - nat_profiler_agent.tool/px_query
 - nat_profiler_agent.tool/response_composer
 - nat_profiler_agent.tool/token_usage
 - nat_profiler_agent/profiler_agent
 - nat_swe_bench.predictors.predict_full.tools/git_repo_tool
 - nat_swe_bench/swe_bench
 - nat_email_phishing_analyzer/email_phishing_analyzer
 - nat.agent.prompt_optimizer/prompt_init
 - nat.agent.prompt_optimizer/prompt_recombiner
 - nat.agent.reasoning_agent/reasoning_agent
 - nat.agent.responses_api_agent/responses_api_agent
 - nat.agent.rewoo_agent/rewoo_agent
 - nat.agent.tool_calling_agent/tool_calling_agent
 - nat.control_flow/sequential_executor
 - nat.control_flow.router_agent/router_agent
 - nat.exper

ValidationError: 3 validation errors for Config
functions.power
  Input tag 'power' found using discriminator() does not match any of the expected tags: '__main__/power', '__main__/square_root', 'nat.agent.react_agent/react_agent', 'nat_profiler_agent.tool/flow_chart', 'nat_profiler_agent.tool/px_query', 'nat_profiler_agent.tool/response_composer', 'nat_profiler_agent.tool/token_usage', 'nat_profiler_agent/profiler_agent', 'nat_swe_bench.predictors.predict_full.tools/git_repo_tool', 'nat_swe_bench/swe_bench', 'nat_email_phishing_analyzer/email_phishing_analyzer', 'nat.agent.prompt_optimizer/prompt_init', 'nat.agent.prompt_optimizer/prompt_recombiner', 'nat.agent.reasoning_agent/reasoning_agent', 'nat.agent.responses_api_agent/responses_api_agent', 'nat.agent.rewoo_agent/rewoo_agent', 'nat.agent.tool_calling_agent/tool_calling_agent', 'nat.control_flow/sequential_executor', 'nat.control_flow.router_agent/router_agent', 'nat.experimental.test_time_compute.functions/execute_score_select_f

## Step 3.4: Run via CLI

Now run the agent using the NAT CLI:


In [ ]:
# Run via CLI (uncomment to execute)
!nat run --config_file configs/math_agent.yaml --input "What is 2 raised to the power of 10?"

In [ ]:
# More CLI examples (uncomment to execute)
# !nat run --config_file configs/math_agent.yaml --input "What is the square root of 256?"
# !nat run --config_file configs/math_agent.yaml --input "Calculate 7 factorial"
!nat run --config_file configs/math_agent.yaml --input "What is the sine of 45 degrees?"
# !nat run --config_file configs/math_agent.yaml --input "What is the mean of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]?"


---

# Summary: The Development Pipeline

## 1️⃣ Prototype (Notebook - Part 1)
- Define config classes (`PowerConfig`, `SquareRootConfig`, etc.)
- Use the **factory pattern**: `NatFunction(config=MyConfig(), registered_function=build_fn)`
- No need to create subclasses - rapid iteration!

## 2️⃣ Export (%%writefile - Part 2)
- Copy function code to `%%writefile` cells
- Add `@register_function` / `@register_function_group` decorators
- Create subclasses for importability: `class Power(PowerConfig, NatFunction)`

## 3️⃣ Deploy (CLI - Part 3)
- Install package: `uv pip install -e .`
- Save configuration: `workflow.save_to_config_file()`
- Run with CLI: `nat run --config_file config.yaml`

## Files Created

| File | Purpose |
|------|---------|
| `math_tools/math_tools/__init__.py` | Package exports |
| `math_tools/math_tools/register.py` | Production function definitions |
| `configs/math_agent.yaml` | Deployment configuration |

## Key Differences: Prototype vs Production

| Aspect | Prototype (Factory Pattern) | Production (Subclass Pattern) |
|--------|-----------|------------|
| **Pattern** | `NatFunction(config=MyConfig())` | `class MyFunc(MyConfig, NatFunction)` |
| **Registration** | `registered_function=build_fn` | `@register_function(config_type=...)` |
| **Location** | Inline in notebook | Separate `.py` file |
| **Importable** | ❌ No class to import | ✅ `from pkg import MyFunc` |
| **CLI Support** | ❌ | ✅ |

## Next Steps

- **[04_functions_and_tools.ipynb](./04_functions_and_tools.ipynb)** - Learn about MCP and A2A tools
- Review the [Functions documentation](../../../docs/source/workflows/functions/index.md)
